In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleeaned_fraud_train.csv")
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,distance_from_prev,time_since_prev_transaction_hours,speed,speed_capped,speed_group,speed_group_order,amt_vs_avg,amt_vs_avg_group,amt_vs_avg_group_order,trans_date
0,2019-01-01 12:47:15,60416207185,"Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,0.000000,0.083333,0.000000,0.000000,Very Low (0–5 km/h),0,0.129767,Much Lower (<0.5x),0,2019-01-01
1,2019-01-02 08:44:57,60416207185,Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,224.769536,19.961667,11.260059,11.260059,Low (5–40 km/h),1,0.944963,Typical (0.9x–1.1x),2,2019-01-02
2,2019-01-02 08:47:36,60416207185,Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,105.220587,0.044167,2382.352919,1000.000000,Extreme (>800 km/h),5,1.465103,Above Usual (1.1x–2x),3,2019-01-02
3,2019-01-02 12:38:14,60416207185,Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,88.152407,3.843889,22.933131,22.933131,Low (5–40 km/h),1,0.620991,Lower (0.5x–0.9x),1,2019-01-02
4,2019-01-02 13:10:46,60416207185,Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,132.876960,0.542222,245.059968,245.059968,High (120–300 km/h),3,0.485155,Much Lower (<0.5x),0,2019-01-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-20 21:04:59,4992346398065154184,"Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,72.134080,8.538889,8.447713,8.447713,Low (5–40 km/h),1,0.891312,Lower (0.5x–0.9x),1,2020-06-20
1296671,2020-06-21 00:41:01,4992346398065154184,Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,50.128303,3.600556,13.922380,13.922380,Low (5–40 km/h),1,1.095015,Typical (0.9x–1.1x),2,2020-06-21
1296672,2020-06-21 02:47:59,4992346398065154184,"Reichert, Rowe and Mraz",shopping_net,246.56,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,94.204046,2.116111,44.517533,44.517533,Moderate (40–120 km/h),2,3.634229,High Spike (2x–5x),4,2020-06-21
1296673,2020-06-21 08:04:28,4992346398065154184,Jewess LLC,shopping_pos,2.62,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,61.101300,5.274722,11.583795,11.583795,Low (5–40 km/h),1,0.038618,Much Lower (<0.5x),0,2020-06-21


# Fraud KPIs

In [2]:
def get_total_transactions(df:pd.DataFrame):
    return len(df)

def get_total_fraud_transactions(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return len(df_fraud)

def get_fraud_rate(df):
    total_fraud = get_total_fraud_transactions(df)
    total = get_total_transactions(df)
    return total_fraud/total * 100

def get_total_sum(df:pd.DataFrame):
    return df['amt'].sum()

def get_fraud_loss_sum(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    return df_fraud["amt"].sum()

def get_fraud_loss_rate(df:pd.DataFrame):
    total_sum = get_total_sum(df)
    fraud_sum = get_fraud_loss_sum(df)
    return fraud_sum/total_sum * 100



In [3]:
get_fraud_loss_sum(df)
get_total_sum(df)

np.float64(91222428.9)

In [4]:
print(get_total_transactions(df), get_total_fraud_transactions(df))

1296675 7506


In [5]:
print(f"fraud rate is {get_fraud_rate(df):.2}%, total loss rate from fraud is {get_fraud_loss_rate(df):.2}%")

fraud rate is 0.58%, total loss rate from fraud is 4.4%


# Time Analysis

In [6]:
def get_fraud_rate_by_hour(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_hour_df = df_fraud.groupby(['hour']).count()["first"]

    hour_df = df.groupby(['hour']).count()["first"]

    fraud_rate_by_hour = fraud_hour_df / hour_df * 100
    
    return fraud_rate_by_hour

def get_fraud_rate_by_day(df:pd.DataFrame):
    df_fraud = df[df["is_fraud"] == 1]
    fraud_week_day_df = df_fraud.groupby(['week_day']).count()["first"]

    hour_df = df.groupby(['week_day']).count()["first"]

    fraud_rate_by_week_day = fraud_week_day_df / hour_df * 100
    
    return fraud_rate_by_week_day

# 3. Category & Merchant Risk


In [7]:


def get_fraud_risk_rate(df, by):
    grouped = df.groupby(by)["is_fraud"].agg(
        total="count",
        frauds="sum"
    ).reset_index()

    grouped["risk"] = grouped["frauds"] / grouped["total"] * 100

    return grouped.sort_values("risk", ascending=False)



In [8]:
category_fraud_risk_rate_df = get_fraud_risk_rate(df,"category")
category_fraud_risk_rate_df

,category,total,frauds,risk
11,shopping_net,97543,1713,1.756149
8,misc_net,63287,915,1.445795
4,grocery_pos,123638,1743,1.409761
12,shopping_pos,116672,843,0.722538
2,gas_transport,131659,618,0.469394
9,misc_pos,79655,250,0.313853
3,grocery_net,45452,134,0.294817
13,travel,40507,116,0.286370
0,entertainment,94014,233,0.247835
10,personal_care,90758,220,0.242403


# 4. Customer Segments

In [9]:
age_group_fraud_risk_rate_df = get_fraud_risk_rate(df, "age_group")    
age_group_fraud_risk_rate_df

,age_group,total,frauds,risk
5,65+,202312,1504,0.743406
4,50-64,267678,1957,0.731102
1,18-24,97491,660,0.676986
2,25-34,278361,1370,0.492167
3,35-49,437403,1955,0.446956
0,0-17,13430,60,0.446761


In [11]:
city_size_fraud_risk_rate_df = get_fraud_risk_rate(df, "city_type") 
city_size_fraud_risk_rate_df

,city_type,total,frauds,risk
2,small city,113165,767,0.677771
1,medium city,32895,215,0.653595
0,large city,32814,194,0.591211
5,village,398055,2299,0.577558
3,small town,509556,2865,0.562254
4,town,210190,1166,0.554736


In [12]:
distance_fraud_risk_rate_df = get_fraud_risk_rate(df, by="distance_group")
distance_fraud_risk_rate_df

,distance_group,total,frauds,risk
3,Same Location (<1 km),106,1,0.943396
0,City-Level (20–100 km),951432,5536,0.581860
1,Far (100–500 km),302853,1747,0.576848
2,Nearby (5–20 km),39686,213,0.536713
4,Very Close (1–5 km),2598,9,0.346420


In [ ]:
get_fraud_risk_rate(df, by="gender")

,gender,total,frauds,risk
1,M,586812,3771,0.642625
0,F,709863,3735,0.526158


In [19]:
get_fraud_risk_rate(df, by="zip")[50:100]

,zip,total,frauds,risk
668,64080,8,8,100.000000
663,63966,8,8,100.000000
398,39562,10,10,100.000000
390,38915,7,7,100.000000
380,37887,11,11,100.000000
95,12508,11,11,100.000000
88,12207,11,11,100.000000
820,78644,9,9,100.000000
815,78208,7,7,100.000000
436,43723,12,12,100.000000


In [32]:
df[df["is_fraud"] == 1].groupby(["state"]).count()[['is_fraud']].sort_values(by="state").reset_index()

,state,is_fraud
0,AK,36
1,AL,215
2,AR,161
3,AZ,37
4,CA,326
5,CO,113
6,CT,16
7,DC,21
8,DE,9
9,FL,281


In [50]:

total_by_state = df.groupby(["state"]).count()[['cc_num']].sort_values(by="state").reset_index()
total_by_state["frauds"] = df[df["is_fraud"] == 1].groupby(["state"]).count()[['is_fraud']].sort_values(by="state").reset_index()["is_fraud"]

total_by_state['rate'] =   total_by_state['frauds']/total_by_state['cc_num']
total_by_state[total_by_state['rate'] < 1]

total_by_state = total_by_state.sort_values(by='rate')

In [56]:
total_by_state["rate"].median()

np.float64(0.005692992443846393)

In [57]:
total_by_state

,state,cc_num,frauds,rate
13,ID,5545,11,0.001984
6,CT,7702,16,0.002077
26,MT,11754,32,0.002722
11,HI,2559,7,0.002735
3,AZ,10770,37,0.003435
28,ND,14786,57,0.003855
18,LA,20965,91,0.004341
31,NJ,24603,118,0.004796
27,NC,30266,149,0.004923
24,MO,38403,191,0.004974


In [58]:
df

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,age_group,city_group,distance,distance_group,mean_amt,amt_std,time_since_prev_transaction,distance_from_prev,time_since_prev_transaction_hours,speed
0,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,25-34,small_town,127.606419,country,56.023366,122.632635,0 days 00:05:00,0.000000,0.083333,0.000000
1,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,25-34,small_town,110.309077,country,56.023366,122.632635,0 days 19:57:42,224.769536,19.961667,11.260059
2,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,25-34,small_town,21.787292,metropolitan_area,56.023366,122.632635,0 days 00:02:39,105.220587,0.044167,2382.352919
3,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,25-34,small_town,87.204338,metropolitan_area,56.023366,122.632635,0 days 03:50:38,88.152407,3.843889,22.933131
4,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,...,25-34,small_town,74.213070,metropolitan_area,56.023366,122.632635,0 days 00:32:32,132.876960,0.542222,245.059968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-20 21:04:59,4992346398065154184,"fraud_Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,50-64,rural,78.492673,metropolitan_area,67.843832,157.223610,0 days 08:32:20,72.134080,8.538889,8.447713
1296671,2020-06-21 00:41:01,4992346398065154184,fraud_Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,50-64,rural,55.400846,metropolitan_area,67.843832,157.223610,0 days 03:36:02,50.128303,3.600556,13.922380
1296672,2020-06-21 02:47:59,4992346398065154184,"fraud_Reichert, Rowe and Mraz",shopping_net,246.56,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,50-64,rural,115.674563,country,67.843832,157.223610,0 days 02:06:58,94.204046,2.116111,44.517533
1296673,2020-06-21 08:04:28,4992346398065154184,fraud_Jewess LLC,shopping_pos,2.62,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,...,50-64,rural,60.513482,metropolitan_area,67.843832,157.223610,0 days 05:16:29,61.101300,5.274722,11.583795
